# RAG 실험 시작 노트북

이 노트북은 **환경 세팅 확인**과 **기본 동작 테스트** 용도입니다.  
청킹·임베딩·DB 구성은 각자 `{이름}_exp01.py`에서 자유롭게 구현하세요.

---

**순서**
1. 환경 확인 (API 키, 패키지)
2. PDF 데이터 불러오기
3. LLM 출력 확인

## 1. 환경 확인

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env"))
api_key = os.getenv("OPENAI_API_KEY", "")
print("API 키:", "OK" if api_key else "없음 — .env 파일 확인")

# PDF 존재 여부
data_dir = Path("../data")
pdfs = list(data_dir.rglob("*.pdf"))
print(f"PDF 파일: {len(pdfs)}개")
for p in pdfs[:5]:
    print(" ", p.relative_to(data_dir))
if len(pdfs) > 5:
    print(f"  ... 외 {len(pdfs)-5}개")

API 키: OK
PDF 파일: 60개
  lg\aircon\AC_FQ16FV6EDN.pdf
  lg\aircon\AC_FQ17FN5BDN.pdf
  lg\aircon\AC_FQ18GC1EHN.pdf
  lg\aircon\AC_FQ18GN7BKN.pdf
  lg\aircon\AC_FQ18GU1BHN.pdf
  ... 외 55개


## 2. PDF 데이터 불러오기

PDF에서 텍스트를 추출하는 가장 기본적인 방법입니다.  
어떻게 나눌지(청킹)는 본인이 정하세요.

In [2]:
import fitz  # PyMuPDF

# 첫 번째 PDF로 테스트
sample_pdf = pdfs[0]
doc = fitz.open(str(sample_pdf))

full_text = "".join(page.get_text() for page in doc)
print(f"파일: {sample_pdf.name}")
print(f"페이지 수: {len(doc)}")
print(f"텍스트 길이: {len(full_text):,}자")
print()
print("--- 앞 500자 미리보기 ---")
print(full_text[:500])

파일: AC_FQ16FV6EDN.pdf


페이지 수: 83
텍스트 길이: 54,951자

--- 앞 500자 미리보기 ---
제품 사용설명서
스탠드 에어컨
모델: LG 휘센 뷰 에어컨(냉방)
MFL71898041
Rev.05_082426
본 사용설명서는 공용으로 제작되어 구입한 제품과 다른 이미지나 내용이 포함되어 있을 수 있습니다.
본 사용설명서는 회사 사정에 따라 변경될 수 있습니다.
목차
안전을 위해 주의하기
4
제품을 사용하기 전에 읽어주세요.
4
안전 기호와 의미
4
R32 냉매 사용 제품( 라벨) 에 표
시되는 안전 기호와 의미
4
경고
5
제품을 설치할 때
5
전원 플러그나 전원선을 다룰 때
7
제품을 사용할 때
7
리모컨을 사용할 때
7
제품을 청소할 때
8
제품 이상 및 고장이 발생했을 때
8
UVnano 기능을 사용할 때
9
만일의 경우를 대비하여
9
주의
9
제품을 설치할 때
9
제품을 사용할 때
LG ThinQ 사용하기
11
LG ThinQ 와 LG 가전 연결하기
11
앱 설치 및 제품 등록하기
11
제품 업그레이드하기
12
와이파이 모듈 사양
12
블루투스 모듈 사양
12
UVnan


## 3. LLM 출력 확인

OpenAI 연결과 답변 형식을 확인합니다.  
실험에서 쓸 `SYSTEM_PROMPT`와 동일한 설정입니다.

In [3]:
import requests

SYSTEM_PROMPT = (
    "너는 가전제품 사용법과 문제 해결을 도와주는 어시스턴트야. 반드시 한국어로만 답해. "
    "아래 검색된 문서 내용만 근거로 자연스럽고 친절하게 답변해. "
    "문서에 없는 내용을 지어내면 안 돼."
)

# ⚠️ 임시: OpenAI 프로젝트 지출 한도(spend limit) 초과로 잠시 로컬(Ollama)로 테스트.
# 한도가 풀리면 아래 셀을 원래 OpenAI 버전으로 되돌릴 것 - 통일 사항(gpt-4o-mini)은
# 최종 제출 실험에서는 반드시 지켜야 한다 (RULES.md 참고).

def generate_local(system_prompt, user_message, model="qwen3.5:9b"):
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
            ],
            "stream": False,
            "think": False,  # think=False 안 하면 응답이 훨씬 느려짐 (예전에 확인함)
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"]


# 임시 컨텍스트로 로컬 LLM 동작 확인
test_context = full_text[:800]
test_query = "이 제품 관련해서 간단히 설명해줘"

answer = generate_local(SYSTEM_PROMPT, f"[문서]\n{test_context}\n\n[질문]\n{test_query}")
print(answer)

네, LG 휘센 뷰 에어컨 (모델명: MFL71898041) 에 대해 소개해 드리겠습니다. 이 제품은 냉방 기능이 주된 목적이며, 안전과 편의를 위해 다양한 주의사항과 설치 방법을 포함하고 있습니다. 또한, UVnano 기능이나 레이더 센서 같은 최신 기술이 적용되어 있는 모델이기도 합니다.

가장 먼저 제품의 안전한 사용을 위해 '안전을 위해 주의하기', '제품을 사용하기 전에 읽어주세요' 등의 내용을 꼭 확인하셔야 합니다. 제품에는 R32 냉매가 사용되고 있어 관련 안전 기호를 주의 깊게 살펴보는 것이 중요합니다.

설치를 하시기 전과使用时中 (사용 중) 에 필요한 점, 리모컨의 사용법 (건전지 끼우기, 연결하기), 그리고 청소 방법 등 기본적인 정보를 문서에 따라 알기 쉽게 안내받으실 수 있습니다. 또한, LG ThinQ 앱을 통해 제품을 연결하고 제어할 수도 있으며, 부가 기능으로 제습, 송풍, 공기청정 등을 세부적으로 설정하여 활용하실 수 있습니다.

만일 제품 이상이나 고장이 발생했다면 대응 방법도 확인해 주시면 좋을 것 같습니다. 궁금한 점이 있으시면 언제든지 말씀해 주세요.


---

세팅이 정상이면 `{이름}_exp01.py`를 열어서 `my_answer()` 구현을 시작하세요.

# 4. 청킹 실험 — LG 에어컨 (1단계: TOC 기반 청킹)

지금까지 위 2번 셀에서 쓴 방식은 "PDF 텍스트 전체를 통째로" 가져온 것뿐이었다.
실제로 청크로 나누는 건 아직 안 했다.

**1단계**: PDF에 내장된 목차(TOC, 북마크)를 이용해서 그 경계로 섹션을 나눠본다.
가장 기본적인 방식만 먼저 넣는다 — 문제해결 챕터 강제 세분화나 긴 섹션 재귀
분할 같은 건 다음 단계에서.</cell_type>


In [4]:
import pymupdf


def build_toc_tree(toc):
    """평평한 [(level, title, page), ...] 목차를 레벨 기준 중첩 트리로 바꾼다."""
    root = []
    stack = [(0, root)]
    for lvl, title, page in toc:
        node = {"level": lvl, "title": title, "page": page, "children": []}
        while len(stack) > 1 and stack[-1][0] >= lvl:
            stack.pop()
        stack[-1][1].append(node)
        stack.append((lvl, node["children"]))
    return root


# 테스트: FQ16FV6EDN 목차를 트리로 바꿔서 상위 2레벨만 출력
doc = fitz.open(str(sample_pdf))
tree = build_toc_tree(doc.get_toc())

def print_tree(nodes, depth=0):
    if depth >= 2:
        return
    for node in nodes:
        print("  " * depth + f"- {node['title']} (p.{node['page']}, 하위 {len(node['children'])}개)")
        print_tree(node["children"], depth + 1)

print_tree(tree)

- 안전을 위해 주의하기 (p.4, 하위 3개)
  - 제품을 사용하기 전에 읽어주세요. (p.4, 하위 2개)
  - 경고 (p.4, 하위 8개)
  - 주의 (p.9, 하위 2개)
- LG ThinQ 사용하기 (p.11, 하위 1개)
  - LG ThinQ와 LG 가전 연결하기 (p.11, 하위 6개)
- 알아보기 (p.13, 하위 1개)
  - 에어컨의 모습과 기능 살펴보기 (p.13, 하위 4개)
- 리모컨으로 사용하기 (p.18, 하위 4개)
  - 리모컨 살펴보기 (p.18, 하위 3개)
  - 냉방 기본 기능 작동하기 (p.26, 하위 5개)
  - 공기청정 기능 사용하기 (p.35, 하위 2개)
  - 추가 기능 및 설정하기 (p.37, 하위 12개)
- 조작부로 사용하기 (p.51, 하위 2개)
  - 조작부 살펴보기 (p.51, 하위 2개)
  - 추가 기능 및 설정하기 (p.53, 하위 2개)
- 관리하기 (p.55, 하위 3개)
  - 청소하기 (p.55, 하위 8개)
  - 냄새 제거하기 (p.70, 하위 1개)
  - 보관하기 (p.71, 하위 1개)
- 고장 신고 전 확인하기 (p.72, 하위 2개)
  - 고장 진단하기 (p.72, 하위 2개)
  - 문제 해결하기 (p.73, 하위 3개)
- 제품 보증서 보기 (p.77, 하위 1개)
  - 제품 보증서 (p.77, 하위 3개)
- 부록 (p.80, 하위 5개)
  - 에너지 효율 등급 (p.80, 하위 1개)
  - 폐가전제품 처리 절차 (p.80, 하위 2개)
  - 오픈소스 정보 (p.80, 하위 1개)
  - 제품 규격 (p.80, 하위 2개)
  - 생활 속 전기안전 캠페인 (p.82, 하위 2개)


In [5]:
def select_nodes(nodes, max_depth):
    """트리에서 max_depth 레벨까지만 청크 경계로 쓴다.
    그 레벨에 자식이 없으면(이미 리프면) 그대로 쓴다."""
    result = []
    for node in nodes:
        if node["level"] < max_depth and node["children"]:
            result.extend(select_nodes(node["children"], max_depth))
        else:
            result.append(node)
    return result


def page_text(doc, start_page_1based, end_page_1based_exclusive):
    start_idx = max(start_page_1based - 1, 0)
    end_idx = min(max(end_page_1based_exclusive - 1, start_idx + 1), len(doc))
    return "\n".join(doc[p].get_text() for p in range(start_idx, end_idx)).strip()


def extract_sections_from_toc(pdf_path, max_depth=2):
    """[{heading, text, page}, ...] - TOC 기반 청킹 1단계 버전.
    (아직 문제해결 강제 세분화, 긴 섹션 재귀 분할, 중복 제목 처리는 없음)"""
    doc = pymupdf.open(pdf_path)
    toc = doc.get_toc()
    if not toc:
        return None  # 이번 단계에서는 TOC 없는 문서는 다루지 않는다

    tree = build_toc_tree(toc)
    boundaries = select_nodes(tree, max_depth=max_depth)

    sections = []
    for i, node in enumerate(boundaries):
        next_page = boundaries[i + 1]["page"] if i + 1 < len(boundaries) else len(doc) + 1
        if next_page <= node["page"]:
            next_page = node["page"] + 1
        text = page_text(doc, node["page"], next_page)
        if text:
            sections.append({"heading": node["title"], "text": text, "page": node["page"]})
    return sections


# 테스트: FQ16FV6EDN 하나로 확인
sections = extract_sections_from_toc(str(sample_pdf))
print(f"섹션 수: {len(sections)}")
for s in sections[:8]:
    print(f"  p.{s['page']:3d} len={len(s['text']):5d}  {s['heading']}")

섹션 수: 22
  p.  4 len=  757  제품을 사용하기 전에 읽어주세요.
  p.  4 len= 4159  경고
  p.  9 len=  942  주의
  p. 11 len= 2438  LG ThinQ와 LG 가전 연결하기
  p. 13 len= 2272  에어컨의 모습과 기능 살펴보기
  p. 18 len= 4397  리모컨 살펴보기
  p. 26 len= 6333  냉방 기본 기능 작동하기
  p. 35 len= 1634  공기청정 기능 사용하기


In [6]:
# LG 에어컨 10개 전체에 돌려서 진단 - 섹션 수 / 최대 길이 / 중복 헤딩 확인
from collections import Counter

aircon_pdfs = sorted(Path("../data/lg/aircon").glob("*.pdf"))
print(f"LG 에어컨 PDF: {len(aircon_pdfs)}개\n")

for p in aircon_pdfs:
    secs = extract_sections_from_toc(str(p))
    lens = [len(s["text"]) for s in secs]
    dup = sum(1 for c in Counter(s["heading"] for s in secs).values() if c > 1)
    print(f"{p.name:22s} 섹션={len(secs):3d}  최대길이={max(lens):6,d}자  중복헤딩={dup}")

LG 에어컨 PDF: 10개



AC_FQ16FV6EDN.pdf      섹션= 22  최대길이= 9,032자  중복헤딩=1


AC_FQ17FN5BDN.pdf      섹션= 22  최대길이= 9,863자  중복헤딩=1


AC_FQ18GC1EHN.pdf      섹션= 23  최대길이= 6,092자  중복헤딩=1


AC_FQ18GN7BKN.pdf      섹션= 22  최대길이= 9,863자  중복헤딩=1


AC_FQ18GU1BHN.pdf      섹션= 23  최대길이= 8,511자  중복헤딩=1


AC_FQ25GN9BKN.pdf      섹션= 22  최대길이= 9,863자  중복헤딩=1


AC_SQ06EJ1WAJ.pdf      섹션= 20  최대길이= 5,328자  중복헤딩=0
AC_SQ07GA3WBN.pdf      섹션= 19  최대길이= 5,320자  중복헤딩=0


AC_SQ07GJ1WEN.pdf      섹션= 19  최대길이= 5,320자  중복헤딩=0


AC_SQ09GK1WEN.pdf      섹션= 19  최대길이= 5,614자  중복헤딩=0


# 5. 2단계 — 중복 헤딩 처리

위에서 발견된 중복 헤딩을 실제로 까보면 전부 `추가 기능 및 설정하기`다.
"리모컨으로 사용하기" 밑(p.37, 리모컨용)과 "조작부로 사용하기" 밑(p.53, 벽 조작부용)에
같은 제목이 두 번 나온다 — 상위 챕터가 다른데 하위 제목만 같은 경우.

**해결**: 직속 부모(레벨1) 제목을 접두어로 붙여서 구분한다.
`[리모컨으로 사용하기] 추가 기능 및 설정하기` vs `[조작부로 사용하기] 추가 기능 및 설정하기`</cell_type>


In [7]:
def select_nodes_with_ancestors(nodes, max_depth, ancestors=()):
    """select_nodes에 조상 경로 추적을 추가한 버전. 각 리프 노드에 '_ancestors'
    (직속 부모부터 최상위까지, 가장 가까운 것이 마지막) 튜플을 붙여서 반환."""
    result = []
    for node in nodes:
        child_ancestors = ancestors + (node["title"],)
        if node["level"] < max_depth and node["children"]:
            result.extend(select_nodes_with_ancestors(node["children"], max_depth, child_ancestors))
        else:
            result.append({**node, "_ancestors": ancestors})
    return result


def disambiguate_titles(sections):
    """같은 heading이 2번 이상 나오면 조상 경로를 필요한 만큼 접두어로 붙인다.
    직속 부모로 안 갈리면 조부모까지 올라간다 (최대 6단계 - 무한루프 방지)."""
    depth = 1
    while True:
        counts = Counter(s["heading"] for s in sections)
        if all(c == 1 for c in counts.values()):
            return
        changed = False
        for s in sections:
            if counts[s["heading"]] > 1:
                ancestors = s.get("_ancestors", ())
                if len(ancestors) < depth:
                    continue
                prefix = " > ".join(ancestors[-depth:])
                base = s.get("_base_heading", s["heading"])
                s["_base_heading"] = base
                s["heading"] = f"[{prefix}] {base}"
                changed = True
        depth += 1
        if not changed or depth > 6:
            return


def extract_sections_v2(pdf_path, max_depth=2):
    """1단계 + 중복 헤딩 처리."""
    doc = pymupdf.open(pdf_path)
    toc = doc.get_toc()
    if not toc:
        return None
    tree = build_toc_tree(toc)
    boundaries = select_nodes_with_ancestors(tree, max_depth=max_depth)

    sections = []
    for i, node in enumerate(boundaries):
        next_page = boundaries[i + 1]["page"] if i + 1 < len(boundaries) else len(doc) + 1
        if next_page <= node["page"]:
            next_page = node["page"] + 1
        text = page_text(doc, node["page"], next_page)
        if text:
            sections.append({
                "heading": node["title"], "text": text, "page": node["page"],
                "_ancestors": node["_ancestors"],
            })

    disambiguate_titles(sections)
    for s in sections:
        s.pop("_ancestors", None)
        s.pop("_base_heading", None)
    return sections


# 테스트: FQ16FV6EDN에서 "추가 기능 및 설정하기"가 어떻게 바뀌었는지 확인
sections_v2 = extract_sections_v2(str(sample_pdf))
for s in sections_v2:
    if "추가 기능" in s["heading"]:
        print(f"  p.{s['page']:3d}  {s['heading']}")

  p. 37  [리모컨으로 사용하기] 추가 기능 및 설정하기


  p. 53  [조작부로 사용하기] 추가 기능 및 설정하기


# 6. 3단계 — 긴 섹션 최종 슬라이싱 (700자 + 15% 오버랩)

`중복 헤딩`은 처리했지만 `냉방 기본 기능 작동하기`(6,333자) 같은 긴 섹션은 아직
그대로다. 이번 단계에서는 (나중에 문제해결 챕터 강제 세분화까지 하기 전에)
가장 단순하게 **일정 길이로 최종 슬라이싱**만 넣어본다.

오버랩을 15% 주는 이유: 경계에서 문장이 뚝 끊기는 걸 완화하기 위해서다
(오버랩 0%로 하면 청크 경계에서 문맥이 끊기는 문제가 있다는 걸 이전에 확인함).

In [8]:
MAX_CHUNK_CHARS = 700
CHUNK_OVERLAP_CHARS = 105  # 700의 15%


def sections_to_chunks(sections, product_model):
    """섹션 리스트를 최종 청크({id, text, metadata})로 슬라이싱한다."""
    step = MAX_CHUNK_CHARS - CHUNK_OVERLAP_CHARS
    chunks = []
    for i, sec in enumerate(sections):
        text = sec["text"]
        pieces = [text[j:j + MAX_CHUNK_CHARS] for j in range(0, len(text), step)] or [text]
        for j, piece in enumerate(pieces):
            chunks.append({
                "id": f"{product_model}_{i}_{j}",
                "text": f"[{sec['heading']}] {piece}",
                "metadata": {"section_title": sec["heading"], "page": sec["page"]},
            })
    return chunks


# 테스트: FQ16FV6EDN으로 전/후 비교
chunks = sections_to_chunks(sections_v2, "FQ16FV6EDN")
chunk_lens = [len(c["text"]) for c in chunks]
print(f"섹션 {len(sections_v2)}개 -> 최종 청크 {len(chunks)}개")
print(f"청크 최대 길이: {max(chunk_lens)}자 (섹션 단계 최대 9,032자였던 것과 비교)")
print()
print("--- 샘플 청크 (냉방 기본 기능 작동하기, 700자 넘던 섹션) ---")
for c in chunks:
    if "냉방 기본 기능" in c["metadata"]["section_title"]:
        print(f"  {c['id']}  len={len(c['text'])}")

섹션 22개 -> 최종 청크 107개
청크 최대 길이: 728자 (섹션 단계 최대 9,032자였던 것과 비교)

--- 샘플 청크 (냉방 기본 기능 작동하기, 700자 넘던 섹션) ---
  FQ16FV6EDN_6_0  len=716
  FQ16FV6EDN_6_1  len=716
  FQ16FV6EDN_6_2  len=716
  FQ16FV6EDN_6_3  len=716
  FQ16FV6EDN_6_4  len=716
  FQ16FV6EDN_6_5  len=716
  FQ16FV6EDN_6_6  len=716
  FQ16FV6EDN_6_7  len=716
  FQ16FV6EDN_6_8  len=716
  FQ16FV6EDN_6_9  len=716
  FQ16FV6EDN_6_10  len=399


# 7. RDB(에러코드) 구축

`data/*_errors.json` 6개(LG/삼성 x 에어컨/냉장고/세탁기)를 SQLite에 시딩한다.

**스키마**:
```
error_solutions (solution_id PK, brand, category, title, content)
error_codes     (code, solution_id FK)
```
한 title에 코드가 여러 개 묶여있는 경우가 많아서(`"CH04 / FL (...)"`) `error_codes`를
별도 테이블로 둬서 코드 하나로 조회하면 바로 조치법을 찾을 수 있게 한다.

**코드 파싱 규칙**: 제목에서 **한글이 포함된 마지막 괄호**를 설명으로 보고, 그 앞부분을
`/`, `,`로 나눠서 코드 목록으로 만든다. LG 냉장고 5개(`Er(E) dH` 같은 표기)는 내부에
괄호가 섞여 있어서 완벽하게 변형 전개는 안 되지만, 정보 손실 없이 원문 그대로
보존한다 (정확한 변형 규칙이 불확실해서 무리하게 풀지 않음).</cell_type>


In [9]:
import re
import sqlite3
import json as jsonlib

ERROR_JSON_FILES = [
    ("../data/lg_aircon_errors.json", "LG"),
    ("../data/lg_fridge_errors.json", "LG"),
    ("../data/lg_washer_errors.json", "LG"),
    ("../data/samsung_aircon_errors.json", "삼성"),
    ("../data/samsung_fridge_errors.json", "삼성"),
    ("../data/samsung_washer_errors.json", "삼성"),
]


def parse_codes(title):
    """제목에서 한글 포함된 마지막 괄호를 설명으로, 그 앞을 코드 목록으로 분리."""
    parens = list(re.finditer(r"\(([^()]*)\)", title))
    desc_match = None
    for m in reversed(parens):
        if re.search(r"[가-힣]", m.group(1)):
            desc_match = m
            break
    if not desc_match:
        return [title.strip()], title
    codes_part = title[: desc_match.start()].strip()
    desc = desc_match.group(1)
    codes = [c.strip() for c in re.split(r"[/,]", codes_part) if c.strip()]
    return codes, desc


def build_error_db(db_path="../data/error_codes.db"):
    conn = sqlite3.connect(db_path)
    conn.execute("DROP TABLE IF EXISTS error_solutions")
    conn.execute("DROP TABLE IF EXISTS error_codes")
    conn.execute("""
        CREATE TABLE error_solutions (
            solution_id INTEGER PRIMARY KEY AUTOINCREMENT,
            brand TEXT NOT NULL,
            category TEXT NOT NULL,
            title TEXT NOT NULL,
            content TEXT NOT NULL
        )
    """)
    conn.execute("""
        CREATE TABLE error_codes (
            code TEXT NOT NULL,
            solution_id INTEGER NOT NULL REFERENCES error_solutions(solution_id),
            UNIQUE(code, solution_id)
        )
    """)

    n_solutions, n_codes = 0, 0
    for path, brand in ERROR_JSON_FILES:
        data = jsonlib.load(open(path, encoding="utf-8"))
        for entry in data:
            category = entry["category"]
            for sec in entry["sections"]:
                codes, _desc = parse_codes(sec["title"])
                cur = conn.execute(
                    "INSERT INTO error_solutions (brand, category, title, content) VALUES (?, ?, ?, ?)",
                    (brand, category, sec["title"], sec["content"]),
                )
                solution_id = cur.lastrowid
                n_solutions += 1
                for code in codes:
                    conn.execute(
                        "INSERT OR IGNORE INTO error_codes (code, solution_id) VALUES (?, ?)",
                        (code, solution_id),
                    )
                    n_codes += 1
    conn.commit()
    print(f"error_solutions {n_solutions}건, error_codes {n_codes}건 저장 완료")
    return conn


conn = build_error_db()

error_solutions 51건, error_codes 127건 저장 완료


In [10]:
def lookup_code(conn, code):
    """코드 하나로 관련 조치법을 전부 조회한다 (브랜드가 달라도 같은 코드면 다 나옴)."""
    rows = conn.execute("""
        SELECT s.solution_id, s.brand, s.category, s.title, s.content
        FROM error_codes c JOIN error_solutions s ON c.solution_id = s.solution_id
        WHERE c.code = ?
    """, (code,)).fetchall()
    return [
        {"solution_id": r[0], "brand": r[1], "category": r[2], "title": r[3], "content": r[4]}
        for r in rows
    ]


# 테스트 1: LG 세탁기 UE (COMMON_QUESTIONS에도 있는 코드)
print("=== UE 조회 ===")
for r in lookup_code(conn, "UE"):
    print(f"  [{r['brand']}/{r['category']}] {r['title']}")

# 테스트 2: 브랜드 겹치는 코드 (LG 세탁기 FE vs 삼성 세탁기 FE)
print("\n=== FE 조회 (LG/삼성 둘 다 있을 것으로 예상) ===")
for r in lookup_code(conn, "FE"):
    print(f"  [{r['brand']}/{r['category']}] {r['title']}")

# 테스트 3: LG 냉장고 보수적으로 등록한 코드 (완벽 전개는 안 했지만 조회는 돼야 함)
print("\n=== 'Er(E) dH' 조회 (보수적으로 등록한 원문 그대로) ===")
for r in lookup_code(conn, "Er(E) dH"):
    print(f"  [{r['brand']}/{r['category']}] {r['title']}")
    print(f"  {r['content'][:100]}")

=== UE 조회 ===
  [LG/세탁기] UE (불균형, 탈수 안됨)
  [삼성/세탁기] UE/UB, U6 (불균형 감지, 언밸런스)

=== FE 조회 (LG/삼성 둘 다 있을 것으로 예상) ===
  [LG/세탁기] FE (과급수 감지)
  [삼성/세탁기] FE/FC (건조팬 동작 안됨, 플렉스워시)

=== 'Er(E) dH' 조회 (보수적으로 등록한 원문 그대로) ===
  [LG/냉장고] Er(E) dH / H / F(r)dH (제상 불량)
  Er(E) dH, H, 또는 F(r)dH 에러는 냉장고가 성에(얼음) 제거 작업을 시작한 이후, 일정 시간이 지나도 내부에서 정상적인 온도가 감지되지 않을 때 표시됩니다. 주요 원


# 8. 4단계 — 문제해결 챕터 강제 세분화

`select_nodes_with_ancestors`는 `max_depth`(=2) 레벨에서 자식이 있어도 멈춘다.
그런데 `문제 해결하기`(레벨2, p.73)는 하위에 `운전`/`소음`/`와이파이` 3개 레벨3
자식이 있는데도, 이 규칙 때문에 전부 하나의 섹션으로 뭉쳐서 나온다 — 확인해보니
`고장 진단하기`(레벨2, p.72)도 `LG ThinQ로 고장 진단하기`/`신호음으로 고장
진단하기` 2개 자식이 같은 이유로 뭉쳐 있었다.

**해결**: 제목이 문제해결 관련 키워드(`TROUBLESHOOTING_KEYWORDS`)와 매칭되면
`max_depth`를 무시하고 강제로 한 단계 더 내려가서 자식들을 개별 섹션으로 뽑는다.

In [11]:
TROUBLESHOOTING_KEYWORDS = ["문제 해결", "고장 진단", "고장 신고", "고장신고", "에러 메시지", "트러블", "고장", "서비스를 요청하기 전에"]
MAX_SECTION_SPAN_PAGES = 20  # 이보다 넓게 계산되면 TOC 손상 의심 -> 안전하게 1페이지로 축소


def _subtree_has_keyword(node):
    """이 노드나 그 하위 어디든 문제해결 키워드가 있으면 True.
    (삼성 일부 PDF는 TOC가 중복 임베드되어 있어서 '서비스를 요청하기 전에'가
    한 단계 더 깊이 들어있는 경우가 있음 - 자기 자신만 보면 그 경우를 놓친다.)"""
    if any(kw in node["title"] for kw in TROUBLESHOOTING_KEYWORDS):
        return True
    return any(_subtree_has_keyword(c) for c in node["children"])


def select_nodes_with_ancestors_v3(nodes, max_depth, ancestors=()):
    """select_nodes_with_ancestors + 문제해결 챕터는 max_depth 무시하고 강제 확장."""
    result = []
    for node in nodes:
        child_ancestors = ancestors + (node["title"],)
        force_expand = _subtree_has_keyword(node)
        if node["children"] and (node["level"] < max_depth or force_expand):
            result.extend(select_nodes_with_ancestors_v3(node["children"], max_depth, child_ancestors))
        else:
            result.append({**node, "_ancestors": ancestors})
    return result


def extract_sections_v3(pdf_path, max_depth=2, toc=None):
    """2단계(중복 헤딩 처리) + 4단계(문제해결 강제 세분화) + TOC 손상 방어(span 캡).
    toc를 직접 넘기면 doc.get_toc() 대신 그걸 쓴다 - PDF에 내장 TOC가 없어도
    인쇄된 목차 페이지를 파싱해서 만든 pseudo-TOC를 그대로 태울 수 있다."""
    doc = pymupdf.open(pdf_path)
    if toc is None:
        toc = doc.get_toc()
    if not toc:
        return None
    tree = build_toc_tree(toc)
    boundaries = select_nodes_with_ancestors_v3(tree, max_depth=max_depth)

    sections = []
    for i, node in enumerate(boundaries):
        next_page = boundaries[i + 1]["page"] if i + 1 < len(boundaries) else len(doc) + 1
        if next_page <= node["page"]:
            next_page = node["page"] + 1
        if next_page - node["page"] > MAX_SECTION_SPAN_PAGES:
            # 삼성 PDF 2개에서 실제로 발견: TOC 북마크가 page=0이거나 중복/오염돼서
            # "다음 경계까지"로 계산하면 문서 대부분(30~40페이지)을 통째로 삼켜버림.
            # 정상적인 섹션은 아무리 길어도 20페이지를 넘지 않는 걸 확인했으므로,
            # 이보다 넓으면 안전하게 1페이지로 줄인다 (해당 페이지 내용은 다른
            # 정상 섹션에서 이미 커버되고 있어 정보 손실이 크지 않음).
            next_page = node["page"] + 1
        text = page_text(doc, node["page"], next_page)
        if text:
            sections.append({
                "heading": node["title"], "text": text, "page": node["page"],
                "_ancestors": node["_ancestors"],
            })

    disambiguate_titles(sections)
    for s in sections:
        s.pop("_ancestors", None)
        s.pop("_base_heading", None)
    return sections


# 테스트: FQ16FV6EDN에서 "고장 신고 전 확인하기" 밑이 어떻게 갈라졌는지 확인
sections_v3 = extract_sections_v3(str(sample_pdf))
print(f"v2 섹션 수: {len(sections_v2)}  ->  v3 섹션 수: {len(sections_v3)}")
print()
for s in sections_v3:
    if 72 <= s["page"] <= 77:
        print(f"  p.{s['page']:3d} len={len(s['text']):5d}  {s['heading']}")

v2 섹션 수: 22  ->  v3 섹션 수: 32

  p. 72 len=  842  LG ThinQ로 고장 진단하기
  p. 72 len=  842  신호음으로 고장 진단하기
  p. 73 len= 2698  운전
  p. 75 len= 1489  소음
  p. 76 len=  963  와이파이
  p. 77 len= 3502  제품 보증서


In [12]:
# LG 에어컨 10개 전체에 v3 돌려서 섹션 수 변화 확인 (v2 대비 몇 개 늘었는지)
for p in aircon_pdfs:
    secs_v2 = extract_sections_v2(str(p))
    secs_v3 = extract_sections_v3(str(p))
    print(f"{p.name:22s} v2={len(secs_v2):3d}  v3={len(secs_v3):3d}  (+{len(secs_v3) - len(secs_v2)})")

AC_FQ16FV6EDN.pdf      v2= 22  v3= 32  (+10)


AC_FQ17FN5BDN.pdf      v2= 22  v3= 34  (+12)
AC_FQ18GC1EHN.pdf      v2= 23  v3= 32  (+9)


AC_FQ18GN7BKN.pdf      v2= 22  v3= 34  (+12)


AC_FQ18GU1BHN.pdf      v2= 23  v3= 32  (+9)


AC_FQ25GN9BKN.pdf      v2= 22  v3= 34  (+12)
AC_SQ06EJ1WAJ.pdf      v2= 20  v3= 30  (+10)


AC_SQ07GA3WBN.pdf      v2= 19  v3= 28  (+9)
AC_SQ07GJ1WEN.pdf      v2= 19  v3= 28  (+9)


AC_SQ09GK1WEN.pdf      v2= 19  v3= 28  (+9)


# 9. 임베딩 + 벡터 DB — LG 에어컨 10개

청킹 로직(v3)이 끝났으니 이제 실제로 임베딩해서 Chroma에 넣는다.

**임베딩 모델**: `dragonkue/BGE-m3-ko` — BGE-M3(다국어, 100개+ 언어 지원)를 한국어
검색 벤치마크(Ko-StrategyQA, AutoRAGRetrieval, MIRACLRetrieval 등)에 맞춰 파인튜닝한
모델. 로컬/무료로 돌아가고, sentence-transformers로 바로 쓸 수 있어서 선택.

**DB 구성**: 브랜드/카테고리별로 컬렉션을 쪼개지 않고 **단일 컬렉션**
(`appliance_manuals`)에 `brand`/`category`/`product_model`을 메타데이터로 넣는다 —
브랜드/카테고리 미지정 질문("에어컨 필터 청소 어떻게 해?")도 컬렉션 하나로 처리
가능하고, 나중에 `where` 필터로 좁혀서 검색할 수 있다.

**시딩 방식**: PDF 한 개 끝날 때마다 그 PDF의 청크를 바로 upsert한다 — 중간에
실패해도 처음부터 다시 하지 않아도 되도록.

In [13]:
import torch
import chromadb
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"임베딩 device: {device}")
embed_model = SentenceTransformer("dragonkue/BGE-m3-ko", device=device)

chroma_client = chromadb.PersistentClient(path="../chroma_db")
# 청킹 로직(TROUBLESHOOTING_KEYWORDS 확장 + span 캡)이 바뀌어서
# 예전 청크 ID가 그대로 남아있으면 안 되니 컬렉션을 통째로 지우고 새로 만든다.
try:
    chroma_client.delete_collection("appliance_manuals")
except Exception:
    pass
collection = chroma_client.create_collection("appliance_manuals")


def product_model_from_path(pdf_path):
    """'AC_FQ16FV6EDN.pdf' -> 'FQ16FV6EDN' (카테고리 접두어 제거)."""
    stem = Path(pdf_path).stem
    return stem.split("_", 1)[1] if "_" in stem else stem


def ingest_pdf(pdf_path, brand, category, extractor=extract_sections_v3):
    """PDF 한 개를 청킹 -> 임베딩 -> upsert. 실패해도 다른 PDF에 영향 없도록 개별 처리."""
    product_model = product_model_from_path(pdf_path)
    sections = extractor(str(pdf_path))
    if not sections:
        print(f"  [건너뜀] TOC 없음: {pdf_path}")
        return 0

    chunks = sections_to_chunks(sections, product_model)
    embeddings = embed_model.encode([c["text"] for c in chunks], normalize_embeddings=True)

    collection.upsert(
        ids=[c["id"] for c in chunks],
        documents=[c["text"] for c in chunks],
        embeddings=embeddings.tolist(),
        metadatas=[
            {
                "brand": brand,
                "category": category,
                "product_model": product_model,
                "section_title": c["metadata"]["section_title"],
                "page": c["metadata"]["page"],
            }
            for c in chunks
        ],
    )
    return len(chunks)


# LG 에어컨 10개 전부 임베딩 + upsert
import time

t0 = time.time()
total_chunks = 0
for p in aircon_pdfs:
    n = ingest_pdf(p, brand="LG", category="에어컨")
    total_chunks += n
    print(f"{p.name:22s} 청크 {n}개 upsert 완료")

elapsed = time.time() - t0
print(f"\n총 {total_chunks}개 청크, 컬렉션 전체 개수: {collection.count()}")
print(f"소요 시간: {elapsed:.1f}초")

임베딩 device: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

AC_FQ16FV6EDN.pdf      청크 120개 upsert 완료


AC_FQ17FN5BDN.pdf      청크 132개 upsert 완료


AC_FQ18GC1EHN.pdf      청크 100개 upsert 완료


...

AC_FQ25GN9BKN.pdf      청크 133개 upsert 완료


AC_SQ06EJ1WAJ.pdf      청크 98개 upsert 완료


AC_SQ07GA3WBN.pdf      청크 87개 upsert 완료


AC_SQ07GJ1WEN.pdf      청크 87개 upsert 완료


AC_SQ09GK1WEN.pdf      청크 90개 upsert 완료

총 1087개 청크, 컬렉션 전체 개수: 1087
소요 시간: 41.4초


In [14]:
# 검색 테스트: COMMON_QUESTIONS 스타일 질문으로 실제 검색해보기
def search(query, top_k=3, where=None):
    query_emb = embed_model.encode([query], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=query_emb, n_results=top_k, where=where)
    candidates = []
    for doc, meta, dist in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
        candidates.append({"text": doc, "metadata": meta, "distance": dist})
    return candidates


for q in ["에어컨 필터 청소는 어떻게 하나요?", "와이파이 연결이 안 될 때 어떻게 해야 하나요?"]:
    print(f"=== 질문: {q} ===")
    for c in search(q, top_k=3):
        print(f"  distance={c['distance']:.4f}  [{c['metadata']['product_model']}/{c['metadata']['section_title']}]")
        print(f"    {c['text'][:80]}...")
    print()

=== 질문: 에어컨 필터 청소는 어떻게 하나요? ===
  distance=0.6021  [FQ18GU1BHN/청소하기]
    [청소하기] 형상이 다를 수 있습니다.
필터 분리하기
1
리모컨의 전원 버튼을 눌러 전원을 끄세요.
필터 이름
이미지
청소 주기
극세 필터
• ...
  distance=0.6593  [SQ06EJ1WAJ/청소하기]
    [청소하기]  청소할 때 극세 필터의 망이 찢어지는 등 
손상되지 않도록 주의하세요. 극세 필터가 손상되면 
이물질이 들어가 제품이 고장 날 수...
  distance=0.6670  [SQ06EJ1WAJ/청소하기]
    [청소하기] 용하면 제품에서 냄새가 날 수 있습니다.
• 청소 주기는 사용 환경이나 시간에 따라 달라질 수 
있습니다. 먼지가 많은 환경에서 제...

=== 질문: 와이파이 연결이 안 될 때 어떻게 해야 하나요? ===
  distance=0.6497  [SQ09GK1WEN/와이파이]
    [와이파이]  해결책...
  distance=0.7832  [FQ18GU1BHN/와이파이]
    [와이파이] 57
고장 신고 전 확인하기 
와이파이
증상
원인 및 해결책
제품과 스마트폰을 
와이파이로 연결할 수 
없어요.
스마트폰에 연결된 ...
  distance=0.7856  [FQ18GC1EHN/와이파이]
    [와이파이] 50
고장 신고 전 확인하기
와이파이
증상
원인 및 해결책
제품과 스마트폰을 
와이파이로 연결할 수 
없어요.
스마트폰에 연결된 와...



# 10. 질문 재구성(Decomposition) + 리랭커

지금까지는 "질문 임베딩 -> Chroma top_k" 한 번으로 끝이었다. 실제 성능을 더
끌어올리려고 두 가지를 추가한다.

**(1) 질문 재구성(Decomposition)**: `"UE 오류랑 필터 청소 방법 같이 알려줘"`처럼
주제가 섞인 복합 질문은 임베딩 하나로는 두 주제 다 잘 못 찾는다. 로컬 LLM
(`qwen3.5:9b`)한테 "주제별로 나눠줘"라고 시켜서 서브 질문 리스트로 쪼갠 다음,
각각 따로 검색해서 합친다. 단일 주제 질문이면 원래 질문 그대로 하나만 돌아온다
(불필요한 LLM 재작성으로 원래 질문 의미가 틀어지는 걸 방지).

**(2) 리랭커**: 임베딩 검색(bi-encoder)은 빠르지만 질문-문서 쌍을 직접 비교하지
않고 각자 벡터화해서 비교하기 때문에 정밀도가 떨어질 수 있다. `dragonkue/bge-reranker-v2-m3-ko`
(cross-encoder, 임베딩 모델과 같은 시리즈의 한국어 파인튜닝 버전)로 후보군을
질문과 직접 비교해서 다시 점수 매기고 정렬한다.

**주의(레이턴시 트레이드오프)**: 둘 다 추가 연산이다 — decomposition은 LLM 호출
1번, 리랭커는 후보 개수만큼 cross-encoder 추론이 추가된다. 예전에 "지금 규모
(top_k=3, 987개 청크)에서는 불필요하다"고 판단해서 뺐었는데, 사용자가 성능을
더 끌어올리고 싶다고 명시적으로 요청해서 이번엔 넣는다 — 대신 GPU로 리랭커를
돌려서 레이턴시를 최대한 줄인다.

In [15]:
import json
import re
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("dragonkue/bge-reranker-v2-m3-ko", device=device)

DECOMPOSE_SYSTEM_PROMPT = (
    "너는 사용자 질문을 검색하기 좋은 형태로 정리하는 어시스턴트야. "
    "질문에 서로 다른 주제(예: 오류코드 문의 + 사용법 문의)가 섞여 있으면 "
    "각 주제를 독립된 질문으로 분리해. 하나의 주제면 원래 질문 그대로 하나만 담아. "
    "반드시 JSON 배열만 출력해. 다른 설명이나 마크다운 없이 [\"질문1\", \"질문2\"] 형식만."
)


def decompose_query(query):
    """복합 질문이면 주제별로 쪼개서 리스트로 반환, 아니면 [query] 그대로."""
    raw = generate_local(DECOMPOSE_SYSTEM_PROMPT, query)
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if not match:
        return [query]
    try:
        parts = json.loads(match.group(0))
        parts = [p.strip() for p in parts if isinstance(p, str) and p.strip()]
        return parts if parts else [query]
    except json.JSONDecodeError:
        return [query]


def rerank(sub_queries, candidates, top_k):
    """cross-encoder로 각 후보를 서브 질문들과 비교하고, 후보당 최고점(max)으로 재정렬.

    처음엔 원본(복합) 질문 그대로 rerank했는데, cross-encoder가 복합 문장에는
    거의 다 0점에 가까운 점수를 매겨서 변별력이 없었다 (실제로 확인함). 분해된
    서브 질문 각각과 비교해서 최고점을 쓰니 제대로 구분됐다 - 애초에 decomposition을
    한 이유가 "주제별로 따로 보자"였으니, rerank도 그 주제 단위로 하는 게 맞다.
    """
    if not candidates:
        return candidates
    for c in candidates:
        sub_scores = reranker.predict([[sq, c["text"]] for sq in sub_queries])
        c["rerank_score"] = float(max(sub_scores))
    return sorted(candidates, key=lambda c: -c["rerank_score"])[:top_k]


def search_v2(query, top_k=3, where=None, candidates_per_subquery=5):
    """decomposition -> 서브 질문별 벡터 검색 -> id 기준 중복 제거 -> 서브 질문 기준 리랭커로 최종 top_k."""
    sub_queries = decompose_query(query)

    merged = {}
    for sub_q in sub_queries:
        for c in search(sub_q, top_k=candidates_per_subquery, where=where):
            chunk_id = c["metadata"]["product_model"] + "_" + c["metadata"]["section_title"] + str(c["metadata"]["page"])
            if chunk_id not in merged or c["distance"] < merged[chunk_id]["distance"]:
                merged[chunk_id] = c

    return rerank(sub_queries, list(merged.values()), top_k=top_k)


# 테스트: 복합 질문으로 확인
compound_q = "UE 오류랑 필터 청소 방법 같이 알려줘"
print(f"질문: {compound_q}")
print(f"  분해: {decompose_query(compound_q)}")
print()
for c in search_v2(compound_q, top_k=4):
    print(f"  rerank_score={c['rerank_score']:.4f}  [{c['metadata']['product_model']}/{c['metadata']['section_title']}]")
    print(f"    {c['text'][:80]}...")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

질문: UE 오류랑 필터 청소 방법 같이 알려줘


  분해: ['UE 오류 코드 확인 및 해결 방법', '어플라이언트 필터 청소 방법']



  rerank_score=0.0021  [FQ18GU1BHN/청소하기]
    [청소하기] 형상이 다를 수 있습니다.
필터 분리하기
1
리모컨의 전원 버튼을 눌러 전원을 끄세요.
필터 이름
이미지
청소 주기
극세 필터
• ...
  rerank_score=0.0016  [SQ09GK1WEN/청소하기]
    [청소하기] 서 제품을 사용할 때는 
필터를 정해진 주기보다 자주 청소하세요.
• 먼지 같은 오염 물질이 많거나 필터가 손상된 경우 새 
필터로...
  rerank_score=0.0013  [FQ18GC1EHN/청소하기]
    [청소하기] 터(옵션)
• 물 세척 
불가능
• 12개월마다 
필터 교체

38
관리하기
2
제품 뒷면에 있는 극세 필터의 손잡이를 잡고 옆으로...
  rerank_score=0.0012  [SQ07GA3WBN/청소하기]
    [청소하기] 은 환경에서 제품을 사용할 때는 
필터를 정해진 주기보다 자주 청소하세요.
• 먼지 같은 오염 물질이 많거나 필터가 손상된 경우 새...


In [16]:
# 레이턴시 비교: 기존 search() vs decomposition+리랭커 search_v2()
import time

test_q = "에어컨 필터 청소는 어떻게 하나요?"

t0 = time.time()
_ = search(test_q, top_k=3)
t_basic = time.time() - t0

t0 = time.time()
_ = search_v2(test_q, top_k=3)
t_v2 = time.time() - t0

print(f"기존 search():     {t_basic:.2f}초")
print(f"search_v2() (decomposition+rerank): {t_v2:.2f}초")
print(f"단일 주제 질문에도 decomposition LLM 호출이 항상 걸리는 게 큰 비중")

기존 search():     0.04초
search_v2() (decomposition+rerank): 5.27초
단일 주제 질문에도 decomposition LLM 호출이 항상 걸리는 게 큰 비중


# 11. LG 냉장고 10개 — 파일 구조가 제각각이라 먼저 진단

에어컨과 달리 냉장고 PDF들은 TOC 구조가 통일돼 있지 않았다. 실제로 10개 전부
확인해보니:

- **8개**: TOC 있음, 레벨 1~4 (에어컨은 최대 3이었는데 냉장고는 4까지 있음).
  `max_depth=2` 기준으로는 레벨 3/4가 상위 섹션에 묶이는 건 에어컨과 동일한 방식이라
  `extract_sections_v3`를 그대로 써도 됨 — 실제로 돌려보니 섹션당 최대 3~4천자,
  중복 헤딩 0개로 문제없이 나왔다.
- **1개(`RF_CA-H17DC.pdf`)**: TOC는 있는데 레벨이 1단계뿐(하위 구조 없음, 14개
  항목). 이것도 기존 로직이 그대로 처리 가능 — 레벨1 노드에 자식이 없으면 자동으로
  리프 취급된다.
- **2개(`RF_S825AW35.pdf`, `RF_S835S31.pdf`)**: **TOC 자체가 없음.** 오래된 모델
  매뉴얼로 보이고, 본문 추출도 단어 사이 공백이 사라져 있는(`"사용전에안전을위한..."`)
  구형 PDF 인코딩이다. 페이지 안에 "차례"(목차) 텍스트가 있긴 한데, 실제로 좌표를
  찍어보니 다단(2~3열) 레이아웃이라 텍스트 읽기 순서가 뒤섞여서(제목과 페이지
  번호가 멀리 떨어져서 나열됨) 정규식으로 안정적으로 파싱하기 어렵다 — 잘못
  파싱해서 엉뚱한 제목에 내용을 붙이느니, 차라리 **페이지 단위로 안전하게
  폴백**한다 (페이지 하나 = 섹션 하나, `heading="N페이지"`). 구조 정보(소제목)는
  잃지만 내용 자체는 전부 보존되고 검색은 여전히 가능하다.

In [17]:
from collections import defaultdict


def _get_page_spans(page):
    """페이지의 텍스트 스팬을 (y, x, 폰트크기, 텍스트) 형태로 뽑는다."""
    d = page.get_text("dict")
    spans = []
    for block in d["blocks"]:
        if "lines" not in block:
            continue
        for line in block["lines"]:
            for span in line["spans"]:
                text = span["text"].strip()
                if text:
                    spans.append({"y": span["bbox"][1], "x": span["bbox"][0], "size": round(span["size"], 1), "text": text})
    return spans


def _page_header_candidates(spans):
    """이 페이지의 본문 크기/소제목 크기를 판정하고, 장식용 초대형 글자는 제외한
    읽기 순서로 정렬된 스팬을 반환한다."""
    if not spans:
        return None, []
    size_counts = Counter(s["size"] for s in spans)
    body_size = size_counts.most_common(1)[0][0]
    # 본문보다 15% 이상 크면 소제목 후보, 12pt 넘으면 장식용 대형 타이틀로 간주해 제외
    header_sizes = {sz for sz in size_counts if body_size * 1.15 <= sz <= 12}
    decorative_sizes = {sz for sz in size_counts if sz > 12}
    ordered = sorted(spans, key=lambda s: (round(s["y"] / 5), s["x"]))
    ordered = [s for s in ordered if s["size"] not in decorative_sizes]
    return header_sizes, ordered


def _merge_adjacent_headers(ordered, header_sizes):
    """같은 컬럼(x 비슷)이고 y도 가까운 헤더 스팬만 "줄바꿈된 한 제목"으로 합친다.
    x가 다르면 격자로 배치된 서로 다른 소제목이므로 합치지 않는다."""
    merged = []
    i = 0
    while i < len(ordered):
        s = ordered[i]
        if s["size"] in header_sizes:
            parts = [s["text"]]
            j = i + 1
            while (j < len(ordered) and ordered[j]["size"] in header_sizes
                   and abs(ordered[j]["y"] - ordered[j - 1]["y"]) < 15
                   and abs(ordered[j]["x"] - ordered[j - 1]["x"]) < 20):
                parts.append(ordered[j]["text"])
                j += 1
            merged.append({"is_header": True, "text": " ".join(parts)})
            i = j
        else:
            merged.append({"is_header": False, "text": s["text"]})
            i += 1
    return merged


def _find_running_labels(doc, min_page_ratio=0.15):
    """여러 페이지에 반복되는 헤더 크기 텍스트(챕터 여백 라벨, 쪽번호 등)를 찾아서
    실제 소제목에서 제외한다."""
    header_page_count = defaultdict(set)
    for pno in range(len(doc)):
        spans = _get_page_spans(doc[pno])
        header_sizes, ordered = _page_header_candidates(spans)
        if not header_sizes:
            continue
        for s in ordered:
            if s["size"] in header_sizes:
                header_page_count[s["text"]].add(pno)
    n_pages = len(doc)
    return {text for text, pages in header_page_count.items() if len(pages) / n_pages >= min_page_ratio}


def _extract_page_subsections(doc, page_index, running_labels):
    """폰트 크기 기반으로 한 페이지를 소제목 단위 서브섹션으로 나눈다.
    소제목 신호가 없으면 None(호출 쪽에서 페이지 전체를 그대로 쓰도록 폴백)."""
    spans = _get_page_spans(doc[page_index])
    header_sizes, ordered = _page_header_candidates(spans)
    if not header_sizes:
        return None
    ordered = [s for s in ordered if not (s["size"] in header_sizes and s["text"] in running_labels)]
    merged = _merge_adjacent_headers(ordered, header_sizes)

    subsections = []
    current_header, current_text = None, []
    for item in merged:
        if item["is_header"]:
            if current_header is not None and current_text:
                subsections.append((current_header, " ".join(current_text)))
            current_header, current_text = item["text"], []
        else:
            current_text.append(item["text"])
    if current_header is not None and current_text:
        subsections.append((current_header, " ".join(current_text)))
    return subsections if subsections else None


def _get_spans(page):
    """페이지의 텍스트 스팬을 (x, y, 폰트크기, 텍스트) 형태로 뽑는다 (인쇄된 목차 파싱용)."""
    d = page.get_text("dict")
    spans = []
    for block in d["blocks"]:
        if "lines" not in block:
            continue
        for line in block["lines"]:
            for span in line["spans"]:
                text = span["text"].strip()
                if text:
                    spans.append({"x": span["bbox"][0], "y": span["bbox"][1], "size": round(span["size"], 1), "text": text})
    return spans


def _cluster_x(values, tol=15):
    """x좌표들을 근접한 값끼리 묶는다 (컬럼 판정용)."""
    vals = sorted(set(values))
    clusters = []
    for v in vals:
        if clusters and v - clusters[-1][-1] <= tol:
            clusters[-1].append(v)
        else:
            clusters.append([v])
    return clusters


def _parse_format_a_toc(page):
    """'찾아보기' 형(신형 삼성) 인쇄 목차 파서. 블록 안에 '제목\t\n페이지\n'이
    반복되는 규칙적인 구조 - 항목 1개짜리 블록은 레벨1(챕터), 여러 개짜리
    블록은 그 챕터의 레벨2 하위항목들로 판정."""
    blocks = page.get_text("blocks")
    results = []
    for b in blocks:
        x0, y0, x1, y1, raw, *_ = b
        lines = [l for l in raw.split("\n") if l.strip()]
        pairs = []
        i = 0
        while i < len(lines) - 1:
            title, num = lines[i].rstrip("\t").strip(), lines[i + 1].strip()
            if title and re.match(r"^\d+$", num) and not re.match(r"^\d+$", title):
                pairs.append((title, int(num)))
                i += 2
            else:
                i += 1
        if not pairs:
            continue
        level = 1 if len(pairs) == 1 else 2
        for title, num in pairs:
            results.append({"level": level, "title": title, "page": num, "x": x0, "y": y0})
    results.sort(key=lambda r: (round(r["x"] / 50), r["y"]))
    return results if results else None


def _parse_format_b_toc(page):
    """'차례' 형(구형) 인쇄 목차 파서. 제목과 페이지번호가 서로 다른 x컬럼에
    떨어져 있어서 y좌표로 매칭해야 한다. 페이지번호는 "가장 많이 쓰인 x컬럼"
    하나만 진짜로 취급하고(장식용 소개 박스의 우연한 y근접 오염 방지),
    레벨(x컬럼)당 항목이 3개 미만이면 노이즈로 간주해 제외한다."""
    spans = _get_spans(page)
    if not spans:
        return None
    num_pat = re.compile(r"^[\d\s]+$")
    text_spans = [s for s in spans if not num_pat.match(s["text"])]
    num_spans = [s for s in spans if num_pat.match(s["text"])]
    if not text_spans or not num_spans:
        return None

    size_counts = Counter(s["size"] for s in text_spans)
    body_size = size_counts.most_common(1)[0][0]
    text_spans = [s for s in text_spans if s["size"] <= body_size * 1.3]

    num_clusters = _cluster_x([s["x"] for s in num_spans], tol=10)
    main_cluster = max(num_clusters, key=len)
    main_num_spans = [s for s in num_spans if round(s["x"]) in {round(v) for v in main_cluster}]
    if len(main_num_spans) < 5:
        return None

    results = []
    for s in text_spans:
        best = None
        for n in main_num_spans:
            dy = abs(n["y"] - s["y"])
            if dy <= 4 and (best is None or dy < best[0]):
                best = (dy, n)
        if best is None:
            continue
        try:
            page_num = int(best[1]["text"].replace(" ", ""))
        except ValueError:
            continue
        results.append({"x": s["x"], "y": s["y"], "text": s["text"], "page": page_num})
    if not results:
        return None

    x_clusters = _cluster_x(sorted({round(r["x"]) for r in results}), tol=15)
    x_centers = [sum(c) / len(c) for c in x_clusters]
    for r in results:
        r["level"] = min(range(len(x_centers)), key=lambda i: abs(x_centers[i] - r["x"])) + 1

    level_counts = Counter(r["level"] for r in results)
    results = [r for r in results if level_counts[r["level"]] >= 3]
    if not results:
        return None
    results.sort(key=lambda r: (round(r["x"] / 50), r["y"]))
    return [{"level": r["level"], "title": r["text"], "page": r["page"]} for r in results]


def get_printed_toc(pdf_path, max_scan_pages=6):
    """PDF 내장 TOC가 없을 때, 인쇄된 목차 페이지("찾아보기" 또는 "차례"/"목차")를
    직접 파싱해서 (level, title, page) 튜플 리스트를 만든다 - build_toc_tree가
    바로 먹을 수 있는 형태. 못 찾으면 None (호출 쪽에서 폰트크기 폴백으로 넘어감).

    두 형식을 시도한다 (실제 5개 TOC 없는 PDF 조사 결과 발견한 두 템플릿):
    - '찾아보기' 형: 여러 페이지에 걸쳐 있을 수 있어서 전부 훑어서 합친다.
    - '차례' 형: 제목/페이지가 다른 컬럼이라 더 까다로움 - 가장 항목이 많이
      나온 단일 페이지 하나만 채택한다 (여러 페이지 조합은 아직 미구현).
    """
    doc = pymupdf.open(pdf_path)
    toc_a = []
    for pno in range(min(max_scan_pages, len(doc))):
        if "찾아보기" in doc[pno].get_text():
            parsed = _parse_format_a_toc(doc[pno])
            if parsed:
                toc_a.extend(parsed)
    if toc_a:
        return [(r["level"], r["title"], r["page"]) for r in toc_a]

    # '차례' 형(Format B)은 일부러 비활성화했다 - 실제로 확인해보니 SRS705IC 같은
    # 구형 매뉴얼은 물리적 PDF 페이지 1장에 인쇄 페이지 번호가 1~2개씩 섞여 있어서
    # (예: 한 물리 페이지에 '9'와 '10'이 같이 찍힘) 목차의 인쇄 페이지 번호를
    # 물리 페이지 인덱스로 그대로 쓰면 문서 후반부에서 에러 없이 조용히 틀린
    # 내용을 가져오는 위험이 있다. Format A(신형 삼성, '찾아보기')는 물리/인쇄
    # 페이지가 1:1로 정확히 맞는 걸 확인해서 그것만 쓴다. Format B를 안전하게
    # 쓰려면 문서 전체를 스캔해서 인쇄번호->물리인덱스 매핑표를 먼저 만들어야
    # 하는데, 영향 범위(3개 파일)에 비해 투입 대비 효과가 낮다고 판단해 보류.
    return None


def extract_sections_by_page(pdf_path):
    """TOC 없는 PDF용 폴백. 페이지 안에 폰트 크기로 구분되는 소제목이 있으면
    그 단위로 쪼개고(예: 한 페이지에 "저온용기"/"계란저장용기"처럼 여러 미니
    주제가 섞여 있던 문제 해결), 신호가 없는 페이지는 안전하게 페이지 전체를
    그대로 섹션 하나로 쓴다 (기존 방식과 동일한 폴백)."""
    doc = pymupdf.open(pdf_path)
    running_labels = _find_running_labels(doc)
    sections = []
    for i, page in enumerate(doc):
        subs = _extract_page_subsections(doc, i, running_labels)
        if subs:
            for header, text in subs:
                if text.strip():
                    sections.append({"heading": header, "text": text.strip(), "page": i + 1})
        else:
            text = page.get_text().strip()
            if text:
                sections.append({"heading": f"{i+1}페이지", "text": text, "page": i + 1})
    return sections


def extract_sections_for_pdf(pdf_path, max_depth=2):
    """TOC 있으면 v3, 없으면 인쇄된 목차 파싱 시도, 그것도 안 되면 페이지/소제목
    폰트크기 폴백. 실제로 확인한 우선순위: 내장 TOC(55개) > 인쇄된 목차 파싱
    가능(3개: SRS705IC, RP20C3111S9, WF21T6500KW) > 폰트크기 폴백(2개: S825AW35,
    S835S31 - 목차가 여러 페이지+세로 사이드바까지 섞여서 더 복잡함, 나중에 개선)."""
    sections = extract_sections_v3(pdf_path, max_depth=max_depth)
    if sections is not None:
        return sections

    printed_toc = get_printed_toc(pdf_path)
    if printed_toc:
        sections = extract_sections_v3(pdf_path, max_depth=max_depth, toc=printed_toc)
        if sections:
            return sections

    return extract_sections_by_page(pdf_path)


# 진단: 냉장고 10개 전부 어느 경로로 처리되는지 확인
fridge_pdfs = sorted(Path("../data/lg/fridge").glob("*.pdf"))
print(f"LG 냉장고 PDF: {len(fridge_pdfs)}개\n")

for p in fridge_pdfs:
    has_toc = extract_sections_v3(str(p)) is not None
    secs = extract_sections_for_pdf(str(p))
    lens = [len(s["text"]) for s in secs]
    method = "TOC 기반(v3)" if has_toc else "페이지/소제목 폴백"
    print(f"{p.name:22s} [{method:16s}] 섹션={len(secs):3d}  최대길이={max(lens):5,d}자")

LG 냉장고 PDF: 10개

RF_B242S32.pdf         [TOC 기반(v3)      ] 섹션= 26  최대길이=3,075자


RF_CA-H17DC.pdf        [TOC 기반(v3)      ] 섹션= 14  최대길이=3,673자


RF_GC-B40BSCQ.pdf      [TOC 기반(v3)      ] 섹션= 38  최대길이=3,318자


RF_GC-B414HG7M.pdf     [TOC 기반(v3)      ] 섹션= 36  최대길이=4,076자


RF_J825MEE042.pdf      [TOC 기반(v3)      ] 섹션= 43  최대길이=3,916자
RF_K135LW123.pdf       [TOC 기반(v3)      ] 섹션= 24  최대길이=3,931자


RF_M402ND.pdf          [TOC 기반(v3)      ] 섹션= 38  최대길이=3,919자


RF_S825AW35.pdf        [페이지/소제목 폴백      ] 섹션=173  최대길이=  596자


RF_S835S31.pdf         [페이지/소제목 폴백      ] 섹션=166  최대길이=  596자


RF_W826AAA492.pdf      [TOC 기반(v3)      ] 섹션= 48  최대길이=3,916자


In [18]:
# LG 냉장고 10개 임베딩 + upsert (같은 단일 컬렉션 appliance_manuals에 추가)
total_chunks_fridge = 0
for p in fridge_pdfs:
    n = ingest_pdf(p, brand="LG", category="냉장고", extractor=extract_sections_for_pdf)
    total_chunks_fridge += n
    print(f"{p.name:22s} 청크 {n}개 upsert 완료")

print(f"\n냉장고 총 {total_chunks_fridge}개 청크, 컬렉션 전체 개수: {collection.count()}")

RF_B242S32.pdf         청크 69개 upsert 완료


RF_CA-H17DC.pdf        청크 38개 upsert 완료


RF_GC-B40BSCQ.pdf      청크 105개 upsert 완료


RF_GC-B414HG7M.pdf     청크 96개 upsert 완료


RF_J825MEE042.pdf      청크 124개 upsert 완료


RF_K135LW123.pdf       청크 69개 upsert 완료


RF_M402ND.pdf          청크 106개 upsert 완료


RF_S825AW35.pdf        청크 174개 upsert 완료


RF_S835S31.pdf         청크 167개 upsert 완료


RF_W826AAA492.pdf      청크 137개 upsert 완료

냉장고 총 1085개 청크, 컬렉션 전체 개수: 2172


In [19]:
# 검색 테스트: 냉장고로 필터링해서 확인 (TOC 폴백된 문서도 검색되는지 포함)
for q in ["냉장고 온도 설정은 어떻게 하나요?", "야채실 관리 방법 알려줘"]:
    print(f"=== 질문: {q} ===")
    for c in search(q, top_k=3, where={"category": "냉장고"}):
        print(f"  distance={c['distance']:.4f}  [{c['metadata']['product_model']}/{c['metadata']['section_title']}]")
        print(f"    {c['text'][:80]}...")
    print()

=== 질문: 냉장고 온도 설정은 어떻게 하나요? ===
  distance=0.5570  [J825MEE042/냉장 & 냉동]
    [냉장 & 냉동] 압기의 램프를 확인하고 켜세요.
냉장, 냉동이 잘 되지 
않아요.
냉장고 설정 온도를 확인했나요?
• 식품의 종류 및 보관 방...
  distance=0.5662  [W826AAA492/냉장 & 냉동]
    [냉장 & 냉동] 으세요.
냉장고 문이 제대로 닫혀 있나요?
• 넣어둔 식품이 문에 끼이지 않았는지 확인하고 문을 꽉 닫으세요.
냉장고 주위에 ...
  distance=0.5763  [B242S32/냉장 & 냉동]
    [냉장 & 냉동] ?
• 넣어둔 식품이 문에 끼이지 않았는지 확인하고 문을 꽉 닫으세요.
냉장고 주위에 적당한 공간이 있나요?
• 냉장고를 설치...

=== 질문: 야채실 관리 방법 알려줘 ===
  distance=0.7204  [S825AW35/10. 신선야채실(하칸)]
    [10. 신선야채실(하칸)] 야채나과일을보관하세요....
  distance=0.7204  [S835S31/10. 신선야채실(하칸)]
    [10. 신선야채실(하칸)] 야채나과일을보관하세요....
  distance=0.7266  [S835S31/신선야채실은]
    [신선야채실은] 내부에물이고이기쉬운곳입니다. 보관식품에물이묻으면상할염려가 있으므로그릇을빼내어물로씻은후깨끗이닦아주세요....



# 12. LG 세탁기 10개

10개 전부 TOC 있음(레벨 1~4, 냉장고와 비슷) — 냉장고처럼 TOC 없는 파일은 없었다.

다만 진단 중에 하나 발견: `WM_FC2521KX6C.pdf`의 `세탁건조 코스 자세히 알아보기`
(p.34)가 하위에 39개 항목(코스 하나하나)을 갖고 있는데 레벨3이라
`TROUBLESHOOTING_KEYWORDS`에 안 걸려서 강제 확장이 안 되고, 8페이지
(19,946자)가 통째로 섹션 하나가 된다. 에러는 아니다 — 기존 슬라이싱
로직(`sections_to_chunks`)이 그대로 700자 단위로 잘라서 처리는 되고 검색도
가능하다. 다만 개별 코스 이름이 청크 제목에 안 붙는 손실은 있다. 지금은 이대로
진행하고, 필요하면 나중에 "너무 긴 섹션은 키워드 상관없이 강제 확장" 같은
일반화된 규칙으로 개선한다.

In [20]:
# LG 세탁기 10개 임베딩 + upsert
washer_pdfs = sorted(Path("../data/lg/washer").glob("*.pdf"))
print(f"LG 세탁기 PDF: {len(washer_pdfs)}개")

total_chunks_washer = 0
for p in washer_pdfs:
    n = ingest_pdf(p, brand="LG", category="세탁기", extractor=extract_sections_for_pdf)
    total_chunks_washer += n
    print(f"{p.name:22s} 청크 {n}개 upsert 완료")

print(f"세탁기 총 {total_chunks_washer}개 청크, 컬렉션 전체 개수: {collection.count()}")

LG 세탁기 PDF: 10개


DR_RH10VTA.pdf         청크 92개 upsert 완료


DR_RH21E5ATKWP.pdf     청크 139개 upsert 완료


ST_SC3GBE50.pdf        청크 81개 upsert 완료


WM_F21VDSK.pdf         청크 108개 upsert 완료


...

WM_FC2521KX6C.pdf      청크 128개 upsert 완료


WM_FC4KC.pdf           청크 93개 upsert 완료


WM_FH23KN.pdf          청크 137개 upsert 완료


WM_T17J4EFNTX.pdf      청크 103개 upsert 완료


WM_TR16MV6.pdf         청크 87개 upsert 완료
세탁기 총 1148개 청크, 컬렉션 전체 개수: 3320


In [21]:
# 검색 테스트: 세탁기로 필터링
for q in ["세탁기 필터 청소는 어떻게 하나요?", "탈수가 안 될 때 어떻게 해야 하나요?"]:
    print(f"=== 질문: {q} ===")
    for c in search(q, top_k=3, where={"category": "세탁기"}):
        print(f"  distance={c['distance']:.4f}  [{c['metadata']['product_model']}/{c['metadata']['section_title']}]")
        print(f"    {c['text'][:80]}...")
    print()

=== 질문: 세탁기 필터 청소는 어떻게 하나요? ===
  distance=0.6200  [FH23KN/제품 청소하기]
    [제품 청소하기] 림이 표시됩니다.
경고
• 필터 청소 및 관리가 제대로 되지 않을 경우, 건조 시간이 
늘어나거나 콘덴서의 먼지 세척이 제대로...
  distance=0.6491  [FH23KN/제품 청소하기]
    [제품 청소하기] 요.
• 물기가 남아 있다면 완전히 말리세요.
• 완전히 마르지 않은 필터를 제품에 장착하면 세탁기 
안에서 냄새가 날 수 있...
  distance=0.6654  [FC2521KX6C/제품 청소하기]
    [제품 청소하기] 쌓여 있을 경우 
제거하세요.
• 보풀이 쌓일 경우 필터 청소 안내 알림이 표시됩니다.
경고
• 필터 청소 및 관리가 제대로 ...

=== 질문: 탈수가 안 될 때 어떻게 해야 하나요? ===
  distance=0.8028  [FH23KN/겨울철 동결 해결하기]
    [겨울철 동결 해결하기] 2
잔수 제거용 호스로 물이 나오지 않는다면 세탁기 
내부에서 동결이 발생했을 가능성이 있습니다. 잔수 
제거용 호스 ...
  distance=0.8188  [FC2521KX6C/겨울철 동결 해결하기]
    [겨울철 동결 해결하기] 호스로 물이 나오지 않는다면 세탁기 
내부에서 동결이 발생했을 가능성이 있습니다. 잔수 
제거용 호스 마개를 다시 닫으...
  distance=0.8249  [RH21E5ATKWP/[관리하기 - 세탁기] 겨울철 동결 해결하기]
    [[관리하기 - 세탁기] 겨울철 동결 해결하기] 을 사용하면 화상을 입을 수 있으니 주의하세요.
2
서비스 커버를 열고 잔수 제거용 호스 마개를...



# 13. 삼성 3종 (에어컨/냉장고/세탁기)

삼성도 미리 진단해봤다:

- 삼성 에어컨 10개: 전부 TOC 있음(레벨 1~2 또는 1~4 섞임), 폴백 필요 없음.
- 삼성 냉장고 10개: 8개 TOC 있음, **2개(`RF_RP20C3111S9.pdf`, `RF_SRS705IC.pdf`) TOC 없음** -> 페이지 폴백.
- 삼성 세탁기 10개: 9개 TOC 있음, **1개(`WM_WF21T6500KW.pdf`) TOC 없음** -> 페이지 폴백.

`extract_sections_for_pdf`가 이미 TOC 유무를 자동으로 분기해서 처리하니 LG 때 만든
코드 그대로 재사용하면 된다. 참고로 `AC_AF19TX978MZ3N.pdf`, `RF_RH82M9152SL.pdf`도
세탁기 `WM_FC2521KX6C.pdf`처럼 레벨3 이하에 큰 하위 목록이 뭉쳐서 섹션 하나가
3~4만자로 큰 경우가 있는데, 슬라이싱은 문제없이 되니 이대로 진행한다.

In [22]:
# 삼성 3종 임베딩 + upsert (같은 단일 컬렉션에 브랜드=삼성으로 추가)
SAMSUNG_CATEGORIES = [("aircon", "에어컨"), ("fridge", "냉장고"), ("washer", "세탁기")]

total_chunks_samsung = 0
for folder, category_kr in SAMSUNG_CATEGORIES:
    pdfs = sorted(Path(f"../data/samsung/{folder}").glob("*.pdf"))
    print(f"--- 삼성 {category_kr} {len(pdfs)}개 ---")
    for p in pdfs:
        n = ingest_pdf(p, brand="삼성", category=category_kr, extractor=extract_sections_for_pdf)
        total_chunks_samsung += n
        print(f"  {p.name:24s} 청크 {n}개 upsert 완료")

print(f"삼성 총 {total_chunks_samsung}개 청크, 컬렉션 전체 개수: {collection.count()}")

--- 삼성 에어컨 10개 ---


  AC_AF17B6474WSN.pdf      청크 67개 upsert 완료


  AC_AF17B7538TZN.pdf      청크 72개 upsert 완료


  AC_AF18RX975CAN.pdf      청크 92개 upsert 완료


  AC_AF19TX978MZ3N.pdf     청크 170개 upsert 완료


...

  ST_DJ40CB9600NE.pdf      청크 81개 upsert 완료


  WM_WA19CG6745BV.pdf      청크 85개 upsert 완료


  WM_WA80F19E8W.pdf        청크 139개 upsert 완료


  WM_WF17M9100KG.pdf       청크 71개 upsert 완료


  WM_WF21T6500KW.pdf       청크 136개 upsert 완료
삼성 총 2785개 청크, 컬렉션 전체 개수: 6105


In [23]:
# 검색 테스트: 삼성으로 필터링, TOC 폴백된 문서 포함해서 확인
for q, where in [
    ("세탁기 통 세척 어떻게 하나요?", {"brand": "삼성"}),
    ("냉장고 정수 필터 교체 방법 알려줘", {"category": "냉장고"}),
]:
    print(f"=== 질문: {q}  (필터: {where}) ===")
    for c in search(q, top_k=3, where=where):
        print(f"  distance={c['distance']:.4f}  [{c['metadata']['brand']}/{c['metadata']['product_model']}/{c['metadata']['section_title']}]")
        print(f"    {c['text'][:80]}...")
    print()

=== 질문: 세탁기 통 세척 어떻게 하나요?  (필터: {'brand': '삼성'}) ===
  distance=0.7751  [삼성/WF17M9100KG/표준세탁 코스]
    [표준세탁 코스] 31
세탁 코스 살펴보기
표준세탁 코스
표준세탁
세탁물에 가장 알맞은 조건을 세탁기가 스스로 정해 자동으로 세탁을 해주는 코스...
  distance=0.7888  [삼성/WA19CG6745BV/세탁 코스 살펴보기]
    [세탁 코스 살펴보기]  세탁물을 넣지 마세요.세제를 넣었을 때 이상 발생이 있을 수 있습니다.
﻿ 설정 
﻿ 기본 설정 
﻿ 선택 가능 
 물...
  distance=0.8085  [삼성/WA80F19E8W/세탁용량 : 21 kg, 23 kg, 25 kg / 물높이 10단 적용 모델]
    [세탁용량 : 21 kg, 23 kg, 25 kg / 물높이 10단 적용 모델] 여 세탁기에 넣고, 세탁기 문을 닫아 주세요.
	-
 세탁 전 ...

=== 질문: 냉장고 정수 필터 교체 방법 알려줘  (필터: {'category': '냉장고'}) ===
  distance=0.4664  [삼성/RS84B5041WW/정수필터 교체하기 (해당 모델만 참조하세요.)]
    [정수필터 교체하기 (해당 모델만 참조하세요.)] 47
청소 및 관리하기
정수필터 교체하기 (해당 모델만 참조하세요.)
냉장고 급수 라인과 연결...
  distance=0.5115  [삼성/RS84B5041WW/정수필터 교체하기 (해당 모델만 참조하세요.)]
    [정수필터 교체하기 (해당 모델만 참조하세요.)] 
ꞏꞏ
하단, 중단, 상단 정수필터 순으로 분리하면 
교체하면서 흘러나오는 물을 줄일 수 있습...
  distance=0.5423  [삼성/RS84B5041WW/정수필터 교체하기 (해당 모델만 참조하세요.)]
    [정수필터 교체하기 (해당 모델만 참조하세요.)] 두 교체하고 나서 Ice Maker 버튼을 
3 초간 눌러주세요.

49
청소 및 관리하기
 ...



# 14. RDB + 벡터 검색 결합 — 최종 answer_query()

지금까지 따로 만든 두 검색 경로를 하나로 합친다.

**라우팅 방식**: 질문 텍스트에서 에러코드처럼 보이는 토큰(영문+숫자 조합, 2자
이상)을 뽑아서 RDB(`error_codes`)에 정확히 일치하는 게 있는지 먼저 확인한다.
있으면 그 조치법을 후보에 추가한다(정확 매칭이라 `distance=0.0`으로 표시 -
`my_answer()` 반환 형식 규칙대로 "없으면 0.0"). 그리고 항상 벡터 검색
(`search_v2` - decomposition+리랭커)도 같이 돌려서 후보에 합친다. 에러코드
매칭이 없어도 벡터 검색 결과는 항상 나오니 폴백이 자연스럽게 된다.

최종적으로 `pipeline/evaluate.py`가 기대하는 `{"answer": str, "candidates":
[{"id", "text", "distance"}]}` 형식으로 반환한다 (CLAUDE.md 규칙).

In [24]:
import re

all_codes = {r[0].upper(): r[0] for r in conn.execute("SELECT DISTINCT code FROM error_codes").fetchall()}


def find_error_codes_in_query(query):
    """질문에서 에러코드로 보이는 토큰(영문+숫자, 2자 이상)을 뽑아 RDB에 존재하는 것만 반환."""
    tokens = re.findall(r"[A-Za-z0-9()]{2,}", query)
    found = []
    for t in tokens:
        if t.upper() in all_codes and all_codes[t.upper()] not in found:
            found.append(all_codes[t.upper()])
    return found


def answer_query(query, top_k=3):
    """RDB(에러코드 정확 매칭) + 벡터 검색(decomposition+리랭커)을 합쳐서 최종 답변 생성."""
    candidates = []

    for code in find_error_codes_in_query(query):
        for r in lookup_code(conn, code):
            candidates.append({
                "id": f"rdb_{r['solution_id']}",
                "text": f"[{r['brand']}/{r['category']}] {r['title']}\n{r['content']}",
                "distance": 0.0,
            })

    for c in search_v2(query, top_k=top_k):
        candidates.append({
            "id": f"{c['metadata']['product_model']}_{c['metadata']['page']}",
            "text": c["text"],
            "distance": c["distance"],
        })

    context = "\n\n".join(c["text"] for c in candidates)
    answer = generate_local(SYSTEM_PROMPT, f"[문서]\n{context}\n\n[질문]\n{query}")

    return {"answer": answer, "candidates": candidates}


# 테스트 1: 순수 에러코드 질문 (RDB 매칭 되는지)
result1 = answer_query("UE 오류가 떴어요 어떻게 해야하나요")
print("=== 테스트 1: UE 오류 ===")
print("후보 id들:", [c["id"] for c in result1["candidates"]])
print("답변:", result1["answer"][:300])

=== 테스트 1: UE 오류 ===
후보 id들: ['rdb_19', 'rdb_47', 'FC4KC_37', 'F21VDSK_43', 'T17J4EFNTX_45']
답변: 안녕하세요, 가전제품 사용법과 문제 해결을 도와드리기 위해 준비된 어시스턴트입니다. 세탁기에서 **UE 에러 (불균형 감지)**가 떳다면 걱정하지 않으셔도大丈夫합니다. 주로 세탁물이 한쪽으로 쏠려 무게 중심이 맞지 않을 때 발생하는 오류이기 때문입니다.

검색된 문서를 바탕으로 상황별로 해결 방법을 순서대로 안내해 드릴게요.

**1 단계: 세탁물 정렬하기**
가장 먼저 할 일은 세탁기를 잠시 멈추는 것입니다 (동작/일시정지 버튼). 세탁물을 한쪽으로 뭉쳐진 상태라면, 이를 골고루 펼쳐서 다시 넣으세요. 특히 이불이나 패딩류처럼 큰


In [25]:
# 테스트 2: 순수 사용법 질문 (RDB 매칭 없이 벡터만)
result2 = answer_query("에어컨 필터 청소는 어떻게 하나요?")
print("=== 테스트 2: 필터 청소 ===")
print("후보 id들:", [c["id"] for c in result2["candidates"]])
print("답변:", result2["answer"][:300])
print()

# 테스트 3: 복합 질문 (에러코드 + 사용법 둘 다)
result3 = answer_query("세탁기 UE 오류랑 필터 청소 방법 같이 알려줘")
print("=== 테스트 3: UE + 필터 청소(복합) ===")
print("후보 id들:", [c["id"] for c in result3["candidates"]])
print("답변:", result3["answer"])

=== 테스트 2: 필터 청소 ===
후보 id들: ['AR06R1130HZN_24', 'AF17B7538TZN_37', 'FQ18GU1BHN_38']
답변: 에어컨 필터를 청소하실 때는 먼저 리모컨의 전원 버튼을 눌러 전원을 끄신 후, 제품 뒷면에 있는 극세 필터 손잡이를 잡고 옆으로 당겨서 필터를 분리하세요. 

청소 방법은 필터의 오염 정도와 종류에 따라 다음과 같습니다:

1. **극세 필터**
   - 오염 물질이 적을 때는 흐르는 물로 씻어주면 됩니다.
   - 먼지가 많이 쌓였다면 중성 세제를 푼 물에 담갔다가 깨끗이 헹구세요.
   - 뜨거운 (40 도 이상) 물을 사용하지 마시고, 세제가 완전히 빠질 때까지 충분히 헹군 뒤 바람이 잘 통하고 그늘진 곳에서 완전히 말리시기 



=== 테스트 3: UE + 필터 청소(복합) ===
후보 id들: ['rdb_19', 'rdb_47', 'WA80F19E8W_46', 'WA19CG6745BV_42', 'DV17T8520BV_57']
답변: 안녕하세요! 세탁기의 'UE' 오류 발생 원인과 해결 방법, 그리고 필터 청소 방법을 친절하게 안내해 드릴게요.

**1. UE 오류 (불균형 감지) 해결 방법**
탈수 중 세탁물이 한쪽으로 쏠려 무게 중심이 맞지 않을 때 발생하므로, 다음 단계를 따라주세요.

*   **세탁물 상태 확인 및 조정:**
    *   세탁기를 멈추고 뭉친 빨랫감을 골고루 펴서 다시 넣어주세요. (세탁통의 2/3 정도만 채우는 것이 탈수 성능에 가장 좋습니다.)
    *   두꺼운 이불류는 김밥처럼 말아 넣으면 균형 잡기에 도움이 됩니다.
    *   세탁망 사용 시 안에 2/3만 채우고 다른 의류 2~3 개를 섞어 넣어 균형을 맞춰주세요.
*   **소자 종류에 따른 코스와 세척 방식:**
    *   이불은 [이불 코스], 패딩류는 [기능성 코스] 로 설정하세요.
    *   베개는 속통을 빼고 커버만 세탁하세요.
*   **수평 확인 및 조절:**
    *   세탁기가 기울어져 있다면 동봉된 스패너로 아래쪽 수평 조정 다리를 조절하여 바닥에 밀착시키고 수평계를 통해 다시 확인해주세요.
*   **비추천 세탁물 주의:**
    *   방수 소재 (스키복, 기저귀 커버 등), 카펫, 발판, 전기장판, 커튼, 고무매트 등을 함께 넣으면 물이 과도하게 머무르거나 무게 중심을 맞출 수 없어 오류가 발생할 수 있으니 피하는 것이 좋습니다.

**2. 필터 분리 및 청소 방법**
청소 후 필터를 다시 끼울 때 주의사항도 꼭 확인해주세요.

*   **세부적인 청소 단계:**
    1.  필터 뚜껑을 열어주세요.
    2.  필터의 윗부분을 아래로 눌러 앞으로 당겨 빼낸 뒤, 남은 찌꺼기를 없애고 깨끗이 닦아주세요.
    3.  다시 끼울 때는 앞면과 뒤면 구분을 